In [ ]:
# !pip install -U bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
from datasets import load_from_disk

In [ ]:
# from datasets import load_from_disk
train_dataset = load_from_disk('/content/drive/MyDrive/cnn_chunks/chunk_0')

In [ ]:
train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 8000
})

In [ ]:
validation_dataset = load_from_disk('/content/drive/MyDrive/cnn_chunks/validation')

In [ ]:
validation_dataset = validation_dataset.shuffle(seed=42).select(range(1000))

In [ ]:
validation_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1000
})

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
import peft
import torch

In [ ]:
model_name="Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    )

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r = 8,
    lora_alpha = 32,
    target_modules = ["q_proj", "v_proj","k_proj","o_proj"],
    lora_dropout = 0.05,
    task_type = TaskType.CAUSAL_LM,
    bias = "none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [ ]:
training_arg = TrainingArguments(
    output_dir = "/content/drive/MyDrive/qwen3_lora_full",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,
    optim = "paged_adamw_32bit",
    save_steps = 100,
    logging_steps = 10,
    learning_rate = 1e-5,
    fp16 = True,
    bf16 = False,
    max_grad_norm = 0.3,
    num_train_epochs =1,
    eval_strategy="steps",
    eval_steps=100,
    gradient_checkpointing=True,
    report_to="none",
    save_strategy="steps",
    save_total_limit=3,
    remove_unused_columns=False,
)

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

In [ ]:
trainer = Trainer(
    model = model,
    args = training_arg,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset,
    data_collator = data_collator,
)

In [ ]:
model.config.use_cache = False  # Disable cache to use gradient checkpointing

NameError: name 'model' is not defined

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
100,3.039841,3.046184
200,2.619352,2.835636
300,2.469334,2.466841
400,2.310617,2.367912
500,2.343228,2.361197


TrainOutput(global_step=500, training_loss=2.60863090133667, metrics={'train_runtime': 5415.597, 'train_samples_per_second': 1.477, 'train_steps_per_second': 0.092, 'total_flos': 3.225648365568e+16, 'train_loss': 2.60863090133667, 'epoch': 1.0})

In [ ]:
save_path = "/content/drive/MyDrive/qwen3_lora_full_model"

trainer.model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/qwen3_lora_full_model/tokenizer_config.json',
 '/content/drive/MyDrive/qwen3_lora_full_model/chat_template.jinja',
 '/content/drive/MyDrive/qwen3_lora_full_model/tokenizer.json')

In [ ]:
from peft import PeftModel

In [ ]:
#the cells bellow will be run in a loop for the chunks created
# everything is same just the training data is changed for each loop

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
ft_model = PeftModel.from_pretrained(
    base_model,
    "/content/drive/MyDrive/qwen3_lora_full_model",
    is_trainable=True   # <-- VERY IMPORTANT
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
ft_model.config.use_cache = False
ft_model.gradient_checkpointing_enable()

In [ ]:
ft_model.print_trainable_parameters()

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


In [ ]:
new_train_dataset = load_from_disk('/content/drive/MyDrive/cnn_chunks/chunk_2')
new_validation_dataset = load_from_disk('/content/drive/MyDrive/cnn_chunks/validation')
new_validation_dataset = new_validation_dataset.shuffle(seed=42).select(range(1000))

In [ ]:
trainer = Trainer(
    model = ft_model,
    args = training_arg,
    train_dataset = new_train_dataset,
    eval_dataset = new_validation_dataset,
    data_collator = data_collator,
)

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
100,2.293633,2.294647
200,2.339265,2.291899
300,2.284078,2.290194
400,2.215906,2.289500
500,2.294935,2.289141


TrainOutput(global_step=500, training_loss=2.286212875366211, metrics={'train_runtime': 4966.9395, 'train_samples_per_second': 1.611, 'train_steps_per_second': 0.101, 'total_flos': 3.225648365568e+16, 'train_loss': 2.286212875366211, 'epoch': 1.0})

In [ ]:
#run this cell only when the training stops in middle to resume.
trainer.train(resume_from_checkpoint="/content/drive/MyDrive/qwen3_lora_full/checkpoint-400")

Step,Training Loss,Validation Loss
500,2.270204,2.299692


TrainOutput(global_step=500, training_loss=0.45933539581298827, metrics={'train_runtime': 1014.6098, 'train_samples_per_second': 7.885, 'train_steps_per_second': 0.493, 'total_flos': 3.225648365568e+16, 'train_loss': 0.45933539581298827, 'epoch': 1.0})

In [ ]:
trainer.model.save_pretrained("/content/drive/MyDrive/qwen3_lora_full_model")